<h1><font color ="Blue"><center> Logistic Regression and GIS</h1>

<h2><font color = "Red"><center> Shaseevarajan Sivanantharajah</h2>

In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import confusion_matrix,classification_report,roc_curve,roc_auc_score,precision_recall_curve
from matplotlib import pyplot as plt
import geopandas as gpd
import numpy as np

In [4]:
path = "/home/harris/Downloads/Shasee"
os.chdir(path)
fname = "NBIFINAL2024.csv"

In [5]:
a = pd.read_csv(fname)
a.head()

,FID,LATDD,LONDD,AGE,RECON,ADT_029,STRUCTURE_KIND_043A,UNSAFE
0,35185702_101560762,35.315839,-101.935450,16.0,0,100,1,0
1,29362100_094272880,29.605833,-94.458000,19.0,0,1,5,0
2,29362940_094254920,29.608167,-94.430333,19.0,0,1,5,0
3,29365160_094324020,29.614333,-94.544500,12.0,0,100,5,0
4,29413300_094044860,29.692500,-94.080167,17.0,1,80,7,1


In [6]:
a['AGE2'] = np.power(a.AGE,2)
a.head()

,FID,LATDD,LONDD,AGE,RECON,ADT_029,STRUCTURE_KIND_043A,UNSAFE,AGE2
0,35185702_101560762,35.315839,-101.935450,16.0,0,100,1,0,256.0
1,29362100_094272880,29.605833,-94.458000,19.0,0,1,5,0,361.0
2,29362940_094254920,29.608167,-94.430333,19.0,0,1,5,0,361.0
3,29365160_094324020,29.614333,-94.544500,12.0,0,100,5,0,144.0
4,29413300_094044860,29.692500,-94.080167,17.0,1,80,7,1,289.0


In [7]:
a.dropna(inplace = True)

In [8]:
len(a)

56514

In [9]:
xcol = ['AGE', 'AGE2','ADT_029','STRUCTURE_KIND_043A', 'RECON']
X = a.copy()
X = X.loc[:,xcol]
Y = a.copy()
Y = Y.loc[:,'UNSAFE']

In [10]:
X.describe()

,AGE,AGE2,ADT_029,STRUCTURE_KIND_043A,RECON
count,56514.000000,56514.000000,56514.000000,56514.000000,56514.000000
mean,37.289397,1892.952773,10830.846604,2.729518,0.194571
std,22.415676,1919.849727,23251.193571,1.887833,0.395874
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18.000000,324.000000,397.000000,1.000000,0.000000
50%,35.000000,1225.000000,2700.500000,1.000000,0.000000
75%,55.000000,3025.000000,10621.500000,5.000000,0.000000
max,124.000000,15376.000000,778093.000000,9.000000,1.000000


In [11]:
Y

0        0
1        0
2        0
3        0
4        1
        ..
56561    0
56562    0
56563    0
56564    0
56565    0
Name: UNSAFE, Length: 56514, dtype: int64

In [12]:
#Train test split
xtrain,xtest,ytrain,ytest = train_test_split(X,Y,test_size = 0.3, stratify = Y, random_state = 10)

In [13]:
model = LR(class_weight='balanced', random_state=20, solver='liblinear')
model.fit(xtrain, ytrain)

LogisticRegression(class_weight='balanced', random_state=20, solver='liblinear')

In [14]:
model.score(xtrain,ytrain)

0.6284031446699866

In [15]:
yscores = model.predict_proba(xtest)[:,1]
yscores

array([0.72367642, 0.34555011, 0.28949901, ..., 0.13686135, 0.18346781,
       0.46057722])

In [16]:
#ROC Curve and AUC
fpr, tpr, thresh = roc_curve(ytest, yscores)
fpr, tpr, thresh 

(array([0.00000000e+00, 0.00000000e+00, 1.28924128e-04, ...,
        9.99677690e-01, 9.99871076e-01, 1.00000000e+00]),
 array([0.00000000e+00, 6.93481276e-04, 6.93481276e-04, ...,
        1.00000000e+00, 1.00000000e+00, 1.00000000e+00]),
 array([       inf, 0.95371529, 0.94078748, ..., 0.04445835, 0.04306881,
        0.04251434]))

In [17]:
auc = roc_auc_score(ytest, yscores)
print("AUC", auc)

AUC 0.7285384688766694


In [18]:
#Optimal Cutoff
J = tpr - fpr
ocutoff = np.argmax(J)
ocutoff = thresh [ocutoff]
ocutoff              

0.35949674674517496

In [19]:
#Create a confusion matrix with a optimum threshold
unsafe = (yscores>=ocutoff).astype(int)
confusion_matrix(ytest,unsafe)

array([[7049, 8464],
       [ 188, 1254]])

In [20]:
sum(unsafe)

9718

In [21]:
sum(ytest)

1442

In [22]:
len(ytest)

16955

In [23]:
len(ytest)-sum(ytest)

15513